Creating Tools

In [ ]:
from ddgs import DDGS
from langchain_core.tools import tool

@tool
def search_tool(query: str) -> str:
    """Search the web for latest news and current information."""
    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=3))
        return "\n".join([f"{r['title']}: {r['body']}" for r in results])

In [14]:
# Creating weather API custom tool for our Agent

import requests
from langchain_core.tools import tool

@tool
def get_weather_data(city: str) -> str:
    """
    This function fetches the current weather data for a given city.
    """
    url = f"https://api.weatherstack.com/current?access_key=efb18722c107f85ee8e00eba1870888a&query={city}"
    response = requests.get(url)
    result = response.json()
    return str(result)

In [20]:
# Creating currency exchange custom tool for our Agent
 
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency : str, target_currency : str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and target currency.
  """
  url = f"https://v6.exchangerate-api.com/v6/1e6847cdee3c21872d964556/pair/{base_currency}/{target_currency}"

  response = requests.get(url)

  return response.json()



@tool
def convert(base_currency : int, conversion_rate : Annotated[float, InjectedToolArg]) -> float:
  """
  Given a currency conversion rate, this function converts a given base currency into the target currency.
  """

  return base_currency * conversion_rate

In [21]:
get_conversion_factor.invoke({"base_currency" : "USD", "target_currency" : "PKR"})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1782172801,
 'time_last_update_utc': 'Tue, 23 Jun 2026 00:00:01 +0000',
 'time_next_update_unix': 1782259201,
 'time_next_update_utc': 'Wed, 24 Jun 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'PKR',
 'conversion_rate': 278.2905}

In [22]:
convert.invoke({"base_currency" : 10, "conversion_rate" : 280.3986})

2803.986

In [16]:
# pip install wikipedia
import wikipedia
from langchain_core.tools import tool

@tool
def get_wikipedia_summary(query: str) -> str:
    """
    Fetches a short factual summary from Wikipedia for a given topic or question.
    Use this for historical facts, famous people, countries, science, etc.
    """
    try:
        summary = wikipedia.summary(query, sentences=4, auto_suggest=True)
        return summary
    except wikipedia.exceptions.DisambiguationError as e:
        return f"Ambiguous query. Try one of these: {e.options[:5]}"
    except wikipedia.exceptions.PageError:
        return f"No Wikipedia page found for '{query}'"

In [ ]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2,
    max_tokens=1024,
)

In [ ]:
import json
from langchain_core.messages import HumanMessage

def run_currency_agent(query: str):
    messages = [HumanMessage(content=query)]

    # Step 1 - LLM decides which tools to call
    ai_message = llm.invoke(messages)
    messages.append(ai_message)

    conversion_rate = None

    # Step 2 - Execute tools manually (inject conversion_rate into convert tool)
    for tool_call in ai_message.tool_calls:

        if tool_call["name"] == "get_conversion_factor":
            tool_msg = get_conversion_factor.invoke(tool_call)
            conversion_rate = json.loads(tool_msg.content)["conversion_rate"]
            messages.append(tool_msg)

        if tool_call["name"] == "convert":
            if conversion_rate:
                tool_call["args"]["conversion_rate"] = conversion_rate
            tool_msg = convert.invoke(tool_call)
            messages.append(tool_msg)

    # Step 3 - LLM gives final answer
    final_response = llm.invoke(messages)
    return final_response.content

# Test
print(run_currency_agent("Convert 10 dollars to PKR"))

In [ ]:
"""
Multi-tool AI agent (web search, weather, currency conversion, Wikipedia)
served through a Streamlit chat UI, powered by Groq + LangGraph.

Local development:
    1. Copy .env.example to .env and fill in your real keys.
    2. pip install -r requirements.txt
    3. streamlit run app.py

Streamlit Cloud deployment:
    1. Push everything EXCEPT .env to GitHub (.gitignore already excludes it).
    2. In your Streamlit Cloud app's Settings -> Secrets, paste the contents
       of .streamlit/secrets.toml.example with your real keys filled in.
"""

import json
import os

import requests
import streamlit as st
import wikipedia
from ddgs import DDGS
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langgraph.prebuilt import create_react_agent

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
MODEL_NAME = "llama-3.1-8b-instant"
APP_TITLE = "Multi-Tool AI Agent"
SYSTEM_PROMPT = """You are a helpful multi-tool assistant.
You have access to the following tools:

1. search_tool - Search the web for latest news and current events
2. get_weather_data - Fetch real-time weather for any city
3. get_conversion_factor - Get currency conversion rate between two currencies
4. convert - Convert an amount using a conversion rate
5. get_wikipedia_summary - Get factual summaries from Wikipedia

Always use the right tool and explain results in simple friendly language."""

st.set_page_config(page_title=APP_TITLE, page_icon="🤖", layout="centered")


# ---------------------------------------------------------------------------
# Secrets handling: works both locally (.env) and on Streamlit Cloud (secrets)
# ---------------------------------------------------------------------------
def get_secret(key: str) -> str | None:
    # st.secrets raises if no secrets.toml exists at all (e.g. local dev
    # using only a .env file), so guard the lookup instead of using `in`.
    try:
        if key in st.secrets:
            return st.secrets[key]
    except Exception:
        pass
    load_dotenv()
    return os.getenv(key)


GROQ_API_KEY = get_secret("GROQ_API_KEY")
WEATHERSTACK_API_KEY = get_secret("WEATHERSTACK_API_KEY")
EXCHANGERATE_API_KEY = get_secret("EXCHANGERATE_API_KEY")

missing = [
    name
    for name, val in [
        ("GROQ_API_KEY", GROQ_API_KEY),
        ("WEATHERSTACK_API_KEY", WEATHERSTACK_API_KEY),
        ("EXCHANGERATE_API_KEY", EXCHANGERATE_API_KEY),
    ]
    if not val
]
if missing:
    st.error(
        "Missing required key(s): "
        + ", ".join(missing)
        + ". Add them to a local `.env` file for development, or to your "
        "Streamlit Cloud app's **Secrets** settings for deployment."
    )
    st.stop()


# ---------------------------------------------------------------------------
# Tools
# ---------------------------------------------------------------------------
@tool
def search_tool(query: str) -> str:
    """Search the web for latest news and current information."""
    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=3))
        return "\n".join(f"{r['title']}: {r['body']}" for r in results)


@tool
def get_weather_data(city: str) -> str:
    """Fetches the current weather data for a given city."""
    url = (
        "https://api.weatherstack.com/current"
        f"?access_key={WEATHERSTACK_API_KEY}&query={city}"
    )
    response = requests.get(url, timeout=10)
    return str(response.json())


@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> str:
    """Fetches the currency conversion factor between a base and target currency."""
    url = (
        f"https://v6.exchangerate-api.com/v6/{EXCHANGERATE_API_KEY}"
        f"/pair/{base_currency}/{target_currency}"
    )
    response = requests.get(url, timeout=10)
    return json.dumps(response.json())


@tool
def convert(base_currency: float, conversion_rate: float) -> float:
    """Given a currency conversion rate, converts a base amount into the target currency."""
    return base_currency * conversion_rate


@tool
def get_wikipedia_summary(query: str) -> str:
    """
    Fetches a short factual summary from Wikipedia for a given topic or question.
    Use this for historical facts, famous people, countries, science, etc.
    """
    try:
        return wikipedia.summary(query, sentences=4, auto_suggest=True)
    except wikipedia.exceptions.DisambiguationError as e:
        return f"Ambiguous query. Try one of these: {e.options[:5]}"
    except wikipedia.exceptions.PageError:
        return f"No Wikipedia page found for '{query}'"


TOOLS = [search_tool, get_weather_data, get_conversion_factor, convert, get_wikipedia_summary]


# ---------------------------------------------------------------------------
# Agent (cached so it's built once per session, not on every message)
# ---------------------------------------------------------------------------
@st.cache_resource
def get_agent():
    llm = ChatGroq(
        model=MODEL_NAME,
        temperature=0.2,
        max_tokens=1024,
        api_key=GROQ_API_KEY,
    )
    return create_react_agent(
        model=llm,
        tools=TOOLS,
        prompt=SystemMessage(content=SYSTEM_PROMPT),
    )


agent = get_agent()


# ---------------------------------------------------------------------------
# Chat state
# ---------------------------------------------------------------------------
if "messages" not in st.session_state:
    st.session_state.messages = []


# ---------------------------------------------------------------------------
# UI
# ---------------------------------------------------------------------------
st.title(f"🤖 {APP_TITLE}")
st.caption(f"Powered by Groq (`{MODEL_NAME}`) + LangGraph · Web search, weather, currency, Wikipedia")

with st.sidebar:
    st.header("Settings")
    st.markdown(
        "**Available tools:**\n"
        "- 🔎 Web search\n"
        "- 🌦️ Weather lookup\n"
        "- 💱 Currency conversion\n"
        "- 📚 Wikipedia summaries"
    )
    if st.button("🗑️ Clear chat", use_container_width=True):
        st.session_state.messages = []
        st.rerun()

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

user_input = st.chat_input("Ask me anything — news, weather, currency, facts...")

if user_input:
    st.session_state.messages.append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.markdown(user_input)

    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            try:
                result = agent.invoke({"messages": [{"role": "user", "content": user_input}]})
                reply = result["messages"][-1].content
            except Exception as e:
                reply = f"⚠️ Something went wrong: {e}"
        st.markdown(reply)

    st.session_state.messages.append({"role": "assistant", "content": reply})

C:\Users\ar261\AppData\Local\Temp\ipykernel_51068\871167832.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


The conversion rate used is based on the current exchange rate at the time of the function call. The result is approximately 2335.0 PKR.
